## Google Colab setup

Set `REPO_URL`, `BRANCH`, `RUN_NAME` and `BASE` in the next cell. Open this notebook from the same GitHub branch. The setup mounts Drive, clones code into the runtime and installs dependencies; local execution skips it. Use a fresh runtime after pushing new code. All notebooks in one experiment use the same run name; choose a new name for a fresh experiment. Run preprocessing and feature engineering to completion before model notebooks. The repository README describes the dataset layout and execution order.


In [1]:
# Configure this cell before running.
#   * Google Colab : edit REPO_URL / BRANCH / RUN_NAME / BASE in the Colab block.
#   * Local machine: edit LOCAL_RUN_NAME / LOCAL_BASE in the local block.
# Use the same RUN_NAME in every notebook of one experiment.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    REPO_URL = "https://github.com/nguyenvanminh281005/bt4222-group12.git"
    BRANCH = "refactor-final"
    RUN_NAME = "run_01"
    BASE = Path("/content/drive/MyDrive/BT4222 - Group 12")
    CHECKOUT = Path("/content/bt4222_checkout")
    drive.mount("/content/drive")
    if not CHECKOUT.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--single-branch",
            REPO_URL, str(CHECKOUT)
        ], check=True)
    active_branch = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "branch", "--show-current"], text=True
    ).strip()
    if active_branch != BRANCH:
        raise ValueError("Checkout branch differs from BRANCH. Use a fresh Colab runtime.")
    CODE_DIR = (CHECKOUT if (CHECKOUT / "utils/data_utils.py").is_file()
                else CHECKOUT / "bt4222-group12")
    if not (CODE_DIR / "utils/data_utils.py").is_file():
        raise FileNotFoundError("The selected branch does not contain the Group 12 project.")
    os.environ.update({
        "BT4222_BASE": str(BASE),
        "MIND_DATA_DIR": str(BASE / "data"),
        "BT4222_PROCESSED_DIR": str(BASE / "processed"),
        "BT4222_FEATURE_DIR": str(BASE / "features"),
        "BT4222_GLOVE_PATH": str(BASE / "data/glove/glove.6B.300d.txt"),
        "BT4222_OUTPUT_DIR": str(BASE / "runs" / RUN_NAME),
    })
    os.chdir(CODE_DIR)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
    ], check=True)
    print("Code:", CODE_DIR)
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])

else:
    # ---- Local machine (VS Code / Jupyter on your own computer) ----
    # LOCAL_BASE must contain train/ and dev/ (news.tsv, behaviors.tsv,
    # entity_embedding.vec) and glove/glove.6B.300d.txt.
    LOCAL_RUN_NAME = "run_02"  # change this for each experiment
    LOCAL_BASE = None  # None -> <repo>/data ; or e.g. Path("/path/to/data")

    CODE_DIR = next(
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "utils" / "data_utils.py").is_file()
    )
    BASE = Path(LOCAL_BASE) if LOCAL_BASE else CODE_DIR / "data"
    # setdefault: variables already exported in the shell take precedence.
    for key, value in {
        "BT4222_BASE": BASE,
        "MIND_DATA_DIR": BASE,
        "BT4222_PROCESSED_DIR": BASE / "processed",
        "BT4222_FEATURE_DIR": BASE / "processed" / "features",
        "BT4222_GLOVE_PATH": BASE / "glove" / "glove.6B.300d.txt",
        "BT4222_OUTPUT_DIR": CODE_DIR / "outputs" / LOCAL_RUN_NAME,
    }.items():
        os.environ.setdefault(key, str(value))
    os.environ.setdefault("BT4222_USE_DRIVE", "0")
    try:
        import torch
        device_name = (torch.cuda.get_device_name(0) if torch.cuda.is_available()
                       else "CPU only")
    except ImportError:
        device_name = "torch not installed"
    print("Mode: local |", device_name)
    print("Code:", CODE_DIR)
    print("Raw data:", os.environ["MIND_DATA_DIR"])
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])

Mode: local | NVIDIA GeForce RTX 5060 Ti
Code: /mnt/data/KHTN2023/Project/bt4222-group12
Raw data: /mnt/data/KHTN2023/Project/bt4222-group12/data
Features: /mnt/data/KHTN2023/Project/bt4222-group12/data/processed/features
Run outputs: /mnt/data/KHTN2023/Project/bt4222-group12/outputs/run_02


# Popularity-Gated NRMS

Learn how much to rely on personal history and historical popularity. A history branch encodes the click sequence; a popularity branch encodes the prepared previous-day category-stratified article set. A learned scalar gate combines the candidate scores:

$$s(u,a)=\alpha_u\,\mathbf u_{hist}^{\top}\mathbf r_a+(1-\alpha_u)\,\mathbf u_{pop}^{\top}\mathbf r_a,\quad 0<\alpha_u<1.$$

An empty history uses the declared learnable empty-history representation. Popularity-set construction and its first-day fallback are explicit in feature preparation. They must not be described as a frozen-training count baseline.

## Experiment design

Configure CE or focal ranking loss, history dropout/truncation, seeds and ablation before execution. Training selects checkpoints on validation nDCG@5. Multi-seed results are measured from actual runs; missing checkpoints are not replaced with numerical defaults. Self-attention ablation retrains the corresponding architecture without self-attention using the same budget.

## Comparisons and interpretation

The optional paired comparison requires matching reference and gated seed checkpoints. A separate global top-20 hard fallback is a post-hoc reference, not the learned two-branch gate. Report overall metrics, history cohorts, gate distributions and temporal stability. Gate weights show branch reliance; neither these weights nor offline logged-slate metrics establish causal engagement gains or unbiased live CTR.


In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)
from utils.metrics import ndcg_at_k, history_groups
import os
import re
import math
import time
import random
import json
from types import SimpleNamespace
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import rankdata
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Configure style
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "default"
)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.size"] = 11
HIST_BINS = [-1, 0, 4, float("inf")]
HIST_LABELS = ["Cold Start (0)", "Sparse (1-4)", "Warm (>=5)"]
_TIE_RNG = np.random.default_rng(0)
from utils.data_utils import source_fingerprint, validate_features, bind_feature_run

feature_id = validate_features(PATHS)
bind_feature_run(PATHS, feature_id)


## 1. Set seed


In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


## 2. Attentionpooling


In [4]:
class AttentionPooling(nn.Module):
    """
    Additive Attention (Attention Pooling) layer as formulated in NRMS (Wu et al., 2019).
    Computes energy a_i = q^T tanh(W * h_i + b), followed by softmax normalization.
    """

    def __init__(self, d_model=256, query_dim=200):
        super().__init__()
        self.proj = nn.Linear(d_model, query_dim)
        self.query = nn.Parameter(torch.empty(query_dim).uniform_(-0.1, 0.1))

    def forward(self, x, mask=None):
        # x: [B, L, D]
        energy = torch.tanh(self.proj(x))  # [B, L, query_dim]
        scores = torch.matmul(energy, self.query)  # [B, L]
        if mask is not None:
            all_pad = (~mask).all(dim=-1, keepdim=True)
            scores = scores.masked_fill(~mask, -1e9)
        alpha = F.softmax(scores, dim=-1)  # [B, L]
        if mask is not None:
            alpha = alpha.masked_fill(all_pad, 0.0)
        out = torch.sum(x * alpha.unsqueeze(-1), dim=1)  # [B, D]
        return out, alpha


## 3. Newsencoder


In [5]:
class NewsEncoder(nn.Module):
    """
    News Encoder:
    Word Embedding (GloVe 300d) -> Projection (256d) -> Multi-Head Self-Attention (16 heads) -> Additive Attention (200d query)
    """

    def __init__(
        self,
        pretrained_embed,
        d_model=256,
        n_heads=16,
        query_dim=200,
        dropout=0.2,
        use_self_attn=True,
    ):
        super().__init__()
        self.use_self_attn = use_self_attn
        self.embedding = nn.Embedding.from_pretrained(
            pretrained_embed, freeze=False, padding_idx=0
        )
        self.dropout = nn.Dropout(dropout)
        self.proj = nn.Linear(pretrained_embed.shape[1], d_model)
        if use_self_attn:
            self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.pooling = AttentionPooling(d_model, query_dim)

    def forward(self, tokens, mask=None, return_weights=False):
        # tokens: [N, L]
        if mask is None:
            mask = tokens > 0
        x = self.embedding(tokens)
        x = self.dropout(x)
        x = self.proj(x)
        if self.use_self_attn:
            key_padding_mask = ~mask
            all_pad = key_padding_mask.all(dim=-1)
            if all_pad.any():
                key_padding_mask = key_padding_mask.clone()
                key_padding_mask[all_pad] = False
            attn_out, _ = self.self_attn(x, x, x, key_padding_mask=key_padding_mask)
            attn_out = self.dropout(attn_out)
        else:
            attn_out = x
        vec, alpha = self.pooling(attn_out, mask=mask)
        if return_weights:
            return vec, alpha
        return vec


## 4. Userencoder


In [6]:
class UserEncoder(nn.Module):
    """
    User Encoder:
    News-level Multi-Head Self-Attention (16 heads) -> Additive Attention (200d query)
    """

    def __init__(
        self, d_model=256, n_heads=16, query_dim=200, dropout=0.2, use_self_attn=True
    ):
        super().__init__()
        self.use_self_attn = use_self_attn
        if use_self_attn:
            self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.pooling = AttentionPooling(d_model, query_dim)

    def forward(self, news_vecs, mask=None, return_weights=False):
        # news_vecs: [B, P, D]
        if self.use_self_attn:
            if mask is not None:
                key_padding_mask = ~mask
                all_pad = key_padding_mask.all(dim=-1)
                if all_pad.any():
                    key_padding_mask = key_padding_mask.clone()
                    key_padding_mask[all_pad] = False
            else:
                key_padding_mask = None
            attn_out, _ = self.self_attn(
                news_vecs, news_vecs, news_vecs, key_padding_mask=key_padding_mask
            )
            attn_out = self.dropout(attn_out)
        else:
            attn_out = news_vecs
        vec, alpha = self.pooling(attn_out, mask=mask)
        if return_weights:
            return vec, alpha
        return vec


## 5. Nrms


In [7]:
class NRMS(nn.Module):
    """
    Complete NRMS Architecture:
    Shared News Encoder + User Encoder + Dot Product Click Predictor

    pop_sets (optional): [n_days, K] news indices of each day's popular set (0 = pad).
    Only used by the hard cold-start fallback at evaluation time.
    """

    def __init__(
        self,
        pretrained_embed,
        news_title_lookup,
        d_model=256,
        n_heads=16,
        query_dim=200,
        dropout=0.2,
        use_self_attn=True,
        pop_sets=None,
    ):
        super().__init__()
        self.register_buffer("news_title_lookup", news_title_lookup)
        self.register_buffer(
            "pop_sets",
            pop_sets if pop_sets is not None else torch.zeros(0, 0, dtype=torch.long),
            persistent=False,
        )
        self.news_encoder = NewsEncoder(
            pretrained_embed,
            d_model,
            n_heads,
            query_dim,
            dropout,
            use_self_attn=use_self_attn,
        )
        self.user_encoder = UserEncoder(
            d_model, n_heads, query_dim, dropout, use_self_attn=use_self_attn
        )

    def forward(self, hist_ids, cand_ids, day_idx=None):
        # hist_ids: [B, P], cand_ids: [B, C]
        B, P = hist_ids.shape
        C = cand_ids.shape[1]
        all_ids = torch.cat([hist_ids.view(-1), cand_ids.view(-1)])
        unique_ids, inv = torch.unique(all_ids, return_inverse=True)
        unique_tokens = self.news_title_lookup[unique_ids]
        unique_vecs = self.news_encoder(unique_tokens)
        all_vecs = unique_vecs[inv]
        hist_news_vecs = all_vecs[: B * P].view(B, P, -1)
        cand_news_vecs = all_vecs[B * P :].view(B, C, -1)
        hist_mask = hist_ids > 0
        user_vecs = self.user_encoder(hist_news_vecs, mask=hist_mask)
        scores = torch.bmm(cand_news_vecs, user_vecs.unsqueeze(2)).squeeze(2)
        return scores


## 6. Gatednrms


In [8]:
class GatedNRMS(NRMS):
    """
    Two-branch NRMS with a learned per-user gate:
      - History branch:    u_hist = UserEncoder(history), or UserEncoder([EMPTY]) if the history is empty
      - Popularity branch: u_pop  = UserEncoder(previous day's category-stratified popular articles)
      - Gate:              alpha  = sigmoid(MLP([u_hist ; u_pop ; log(1+|H|) / log(1+P)]))
      - Score:             s(c)   = alpha * <u_hist, e_c> + (1 - alpha) * <u_pop, e_c>
    No hand-set threshold: the [EMPTY] vector lets the gate learn when to trust the popularity branch.
    """

    def __init__(
        self,
        pretrained_embed,
        news_title_lookup,
        d_model=256,
        n_heads=16,
        query_dim=200,
        dropout=0.2,
        use_self_attn=True,
        pop_sets=None,
        gate_hidden=128,
        max_hist_len=50,
    ):
        super().__init__(
            pretrained_embed,
            news_title_lookup,
            d_model,
            n_heads,
            query_dim,
            dropout,
            use_self_attn=use_self_attn,
            pop_sets=pop_sets,
        )
        self.max_hist_len = max_hist_len
        self.empty_vec = nn.Parameter(torch.empty(d_model).normal_(0.0, 0.1))
        self.gate = nn.Sequential(
            nn.Linear(2 * d_model + 1, gate_hidden),
            nn.ReLU(),
            nn.Linear(gate_hidden, 1),
        )

    def encode_popular(self, pop_news_vecs):
        # pop_news_vecs: [n_days, K, D] -> u_pop per day: [n_days, D]
        return self.user_encoder(pop_news_vecs, mask=(self.pop_sets > 0))

    def gate_users(self, hist_news_vecs, hist_mask, u_pop):
        # hist_news_vecs: [B, P, D], hist_mask: [B, P], u_pop: [B, D]
        B, P, D = hist_news_vecs.shape
        n_hist = hist_mask.sum(dim=1)
        empty = n_hist == 0
        # Empty history -> a single learnable [EMPTY] item in the last slot
        slot = torch.zeros_like(hist_mask)
        slot[:, -1] = True
        fill = slot & empty.unsqueeze(1)
        hist_news_vecs = torch.where(
            fill.unsqueeze(-1), self.empty_vec.expand(B, P, D), hist_news_vecs
        )
        hist_mask = hist_mask | fill
        u_hist = self.user_encoder(hist_news_vecs, mask=hist_mask)
        len_feat = (
            torch.log1p(n_hist.float()) / math.log1p(self.max_hist_len)
        ).unsqueeze(1)
        alpha = torch.sigmoid(
            self.gate(torch.cat([u_hist, u_pop, len_feat], dim=1))
        ).squeeze(1)
        return u_hist, alpha

    def forward(self, hist_ids, cand_ids, day_idx=None, return_alpha=False):
        B, P = hist_ids.shape
        C = cand_ids.shape[1]
        n_days, K = self.pop_sets.shape
        all_ids = torch.cat(
            [hist_ids.view(-1), cand_ids.view(-1), self.pop_sets.view(-1)]
        )
        unique_ids, inv = torch.unique(all_ids, return_inverse=True)
        all_vecs = self.news_encoder(self.news_title_lookup[unique_ids])[inv]
        hist_news_vecs = all_vecs[: B * P].view(B, P, -1)
        cand_news_vecs = all_vecs[B * P : B * P + B * C].view(B, C, -1)
        pop_news_vecs = all_vecs[B * P + B * C :].view(n_days, K, -1)
        u_pop = self.encode_popular(pop_news_vecs)[day_idx]  # [B, D]
        u_hist, alpha = self.gate_users(hist_news_vecs, hist_ids > 0, u_pop)
        s_hist = torch.bmm(cand_news_vecs, u_hist.unsqueeze(2)).squeeze(2)
        s_pop = torch.bmm(cand_news_vecs, u_pop.unsqueeze(2)).squeeze(2)
        scores = alpha.unsqueeze(1) * s_hist + (1 - alpha.unsqueeze(1)) * s_pop
        if return_alpha:
            return scores, alpha
        return scores


## 7. Traindataset


In [9]:
class TrainDataset(Dataset):
    """
    History dropout (training only): with prob p_empty the history is wiped (simulated cold-start user),
    with prob p_trunc it is cut to its last 1..max_trunc clicks (simulated sparse user).
    """

    def __init__(self, samples, p_empty=0.0, p_trunc=0.0, max_trunc=3):
        self.samples = samples
        self.p_empty = p_empty
        self.p_trunc = p_trunc
        self.max_trunc = max_trunc

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        h, c, d = self.samples[idx]
        r = random.random() if (self.p_empty > 0 or self.p_trunc > 0) else 1.0
        if r < self.p_empty:
            h = [0] * len(h)
        elif r < self.p_empty + self.p_trunc:
            valid = [x for x in h if x > 0]
            k = random.randint(1, self.max_trunc)
            if len(valid) > k:
                h = [0] * (len(h) - k) + valid[-k:]
        return (
            torch.tensor(h, dtype=torch.long),
            torch.tensor(c, dtype=torch.long),
            torch.tensor(d, dtype=torch.long),
        )


## 8. Ranking loss


In [10]:
def ranking_loss(scores, loss_type="ce", gamma=2.0):
    """
    Listwise loss over [1 positive, K negatives] (positive at index 0).
      ce    : -log p+                    (sampled softmax, NRMS default)
      focal : -(1 - p+)^gamma * log p+   (down-weights easy impressions)
    """
    log_p = F.log_softmax(scores, dim=-1)[:, 0]
    if loss_type == "focal":
        return -(((1.0 - log_p.exp()) ** gamma) * log_p).mean()
    return -log_p.mean()


## 9. Eval impression


In [11]:
def eval_impression(labels, scores):
    """
    Evaluates AUC, MRR, NDCG@5, and NDCG@10 for a single impression slate.
    Ties are not resolved by the logged slate order (a position artifact): AUC uses average ranks,
    so fully tied scores give exactly 0.5, and MRR/NDCG break ties uniformly at random.
    """
    labels = np.asarray(labels, dtype=np.int32)
    scores = np.asarray(scores, dtype=np.float32)
    pos_mask = labels == 1
    n_pos = pos_mask.sum()
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        auc = 0.5
    else:
        ranks = rankdata(scores)
        auc = (ranks[pos_mask].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg)
    perm = _TIE_RNG.permutation(len(scores))
    desc_order = perm[np.argsort(-scores[perm], kind="stable")]
    sorted_labels = labels[desc_order]
    pos_ranks = np.where(sorted_labels == 1)[0]
    mrr = 1.0 / (pos_ranks[0] + 1) if len(pos_ranks) > 0 else 0.0
    return (
        float(auc),
        float(mrr),
        ndcg_at_k(labels, sorted_labels, 5),
        ndcg_at_k(labels, sorted_labels, 10),
    )


## 10. Evaluate model


In [12]:
@torch.no_grad()
def evaluate_model(
    model, eval_records, news_title_lookup, device, batch_size=2048, cold_fallback=False
):
    """
    Vectorized high-throughput evaluation:
    1. Pre-encodes all news articles once with NewsEncoder.
    2. Encodes all user representations in batches. For GatedNRMS the effective user vector is
       alpha * u_hist + (1 - alpha) * u_pop, which is exactly equivalent to mixing the two branch scores.
       For NRMS with cold_fallback=True, users with empty history get u_pop (hard fallback baseline).
    3. Computes candidate dot products and ranking metrics per impression.
    """
    model.eval()
    num_news = len(news_title_lookup)
    news_vecs = torch.zeros(num_news, 256, device=device)
    for i in range(0, num_news, batch_size):
        chunk = news_title_lookup[i : min(i + batch_size, num_news)].to(device)
        vecs = model.news_encoder(chunk)
        news_vecs[i : min(i + batch_size, num_news)] = vecs
    news_vecs[0] = 0.0  # Pad news
    gated = isinstance(model, GatedNRMS)
    use_pop = gated or (cold_fallback and model.pop_sets.numel() > 0)
    if use_pop:
        pop_sets = model.pop_sets
        u_pop_all = model.user_encoder(
            news_vecs[pop_sets], mask=(pop_sets > 0)
        )  # [n_days, D]
    num_eval = len(eval_records)
    hist_matrix = torch.zeros(num_eval, 50, dtype=torch.long, device=device)
    for idx, rec in enumerate(eval_records):
        hist_matrix[idx] = torch.tensor(rec["history"], dtype=torch.long, device=device)
    day_vec = torch.tensor(
        [rec.get("day_idx", 0) for rec in eval_records], dtype=torch.long, device=device
    )
    user_vecs = torch.zeros(num_eval, 256, device=device)
    alphas = torch.ones(num_eval, device=device)
    user_batch_size = 1024
    for i in range(0, num_eval, user_batch_size):
        j = min(i + user_batch_size, num_eval)
        chunk_hist = hist_matrix[i:j]
        chunk_news_vecs = news_vecs[chunk_hist]  # [B, 50, 256]
        mask = chunk_hist > 0
        if gated:
            u_pop = u_pop_all[day_vec[i:j]]
            u_hist, alpha = model.gate_users(chunk_news_vecs, mask, u_pop)
            u_vecs = alpha.unsqueeze(1) * u_hist + (1 - alpha.unsqueeze(1)) * u_pop
            alphas[i:j] = alpha
        else:
            u_vecs = model.user_encoder(chunk_news_vecs, mask=mask)
            if use_pop:
                empty = ~mask.any(dim=1)
                u_vecs = torch.where(
                    empty.unsqueeze(1), u_pop_all[day_vec[i:j]], u_vecs
                )
                alphas[i:j] = (~empty).float()
        user_vecs[i:j] = u_vecs
    alphas = alphas.cpu().numpy()
    aucs, mrrs, ndcg5s, ndcg10s = [], [], [], []
    detailed_results = []
    for i, rec in enumerate(eval_records):
        cands = rec["candidates"]
        labels = rec["labels"]
        u_vec = user_vecs[i]
        c_vecs = news_vecs[cands]
        scores = torch.mv(c_vecs, u_vec).cpu().numpy()
        auc, mrr, n5, n10 = eval_impression(labels, scores)
        aucs.append(auc)
        mrrs.append(mrr)
        ndcg5s.append(n5)
        ndcg10s.append(n10)
        detailed_results.append(
            {
                "impression_id": rec["impression_id"],
                "timestamp": rec["timestamp"],
                "history_len": rec["history_len"],
                "hour": (
                    rec["timestamp"].hour if hasattr(rec["timestamp"], "hour") else 0
                ),
                "alpha": float(alphas[i]),
                "auc": auc,
                "mrr": mrr,
                "ndcg5": n5,
                "ndcg10": n10,
            }
        )
    return {
        "auc": float(np.mean(aucs)),
        "mrr": float(np.mean(mrrs)),
        "ndcg5": float(np.mean(ndcg5s)),
        "ndcg10": float(np.mean(ndcg10s)),
        "detailed": detailed_results,
    }


## 11. Metrics of


In [13]:
def metrics_of(res):
    return {k: float(res[k]) for k in ["auc", "mrr", "ndcg5", "ndcg10"]}


## 12. Stratify by history


In [14]:
def stratify_by_history(detailed_df):
    detailed_df = detailed_df.copy()
    detailed_df["history_bin"] = pd.cut(
        detailed_df["history_len"], bins=HIST_BINS, labels=HIST_LABELS
    )
    out = {}
    for b in HIST_LABELS:
        sub = detailed_df[detailed_df["history_bin"] == b]
        out[b] = {
            "count": int(len(sub)),
            "auc": float(sub["auc"].mean()),
            "mrr": float(sub["mrr"].mean()),
            "ndcg5": float(sub["ndcg5"].mean()),
            "ndcg10": float(sub["ndcg10"].mean()),
            "alpha_mean": float(sub["alpha"].mean()),
            "alpha_std": float(sub["alpha"].std()) if len(sub) > 1 else 0.0,
        }
    return out


## 13. Experiment configuration


In [15]:
# Choose the configuration to train. Reproducing the report needs TWO runs of
# this notebook with the same RUN_NAME:
#   1) NRMS_VARIANT = "nrms"   -> plain NRMS reference, seeds 42/43/44, ablation, hard fallback
#   2) NRMS_VARIANT = "gated"  -> Gated NRMS (history dropout/truncation p=0.1), seeds 42/43/44, ablation
# The paired multi-seed comparison (section 14) runs automatically once both
# sets of checkpoints exist. "gated_nodrop" and "gated_focal" are optional
# single-seed variants reported in reports/results/.
NRMS_VARIANT = "gated"

VARIANT_PRESETS = {
    "nrms": dict(model="nrms", loss="ce", hist_dropout=0.0, hist_trunc=0.0,
                 skip_seeds=False, skip_ablation=False),
    "gated": dict(model="gated", loss="ce", hist_dropout=0.1, hist_trunc=0.1,
                  skip_seeds=False, skip_ablation=False),
    "gated_nodrop": dict(model="gated", loss="ce", hist_dropout=0.0, hist_trunc=0.0,
                         skip_seeds=True, skip_ablation=True),
    "gated_focal": dict(model="gated", loss="focal", hist_dropout=0.1, hist_trunc=0.1,
                        skip_seeds=True, skip_ablation=True),
}
config = SimpleNamespace(
    tag=NRMS_VARIANT,
    eval_only=False,
    epochs=3,
    batch_size=128,
    lr=1e-4,
    num_neg=4,
    gamma=2.0,
    pop_per_cat=2,
    pop_extra=10,
    pop_min_cat_clicks=20,
    debug=False,
    **VARIANT_PRESETS[NRMS_VARIANT],
)
models_dir = str(PATHS["outputs"] / "nrms")
checkpoints_dir = os.path.join(models_dir, "checkpoints")
figures_dir = os.path.join(models_dir, "figures")
tag = config.tag or config.model

if config.debug:
    tag = f"debug_{tag}"

file_prefix = "" if tag == "nrms" else f"{tag}_"
summary_path = os.path.join(
    figures_dir,
    "results_summary.json" if tag == "nrms" else f"results_summary_{tag}.json",
)
os.makedirs(checkpoints_dir, exist_ok=True)
os.makedirs(figures_dir, exist_ok=True)
print("=" * 70)
print(
    f"BT4222 Group 12: {config.model.upper()} Training & Comprehensive Evaluation (run tag: {tag})"
)
print(f"Config: {vars(config)}")
print("=" * 70)
set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(
    f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})"
)
from types import SimpleNamespace

payload = pd.read_pickle(PATHS["features"] / "nrms_features.pkl")
data = SimpleNamespace(**payload)
assert data.popularity_config == {
    "per_cat": config.pop_per_cat,
    "extra": config.pop_extra,
    "min_cat_clicks": config.pop_min_cat_clicks,
}, "Prepare matching popularity inputs in feature engineering."
assert not config.debug, "Use a separate prepared fixture dataset for smoke testing."
news_id2idx = data.news_id2idx
news_idx2id = {idx: nid for nid, idx in news_id2idx.items()}
news_title_lookup = torch.as_tensor(
    data.news_title_lookup, dtype=torch.long, device=device
)
pretrained_embeddings = torch.as_tensor(data.pretrained_embeddings, dtype=torch.float32)
all_news = data.all_news
train_df, val_df, test_df = data.train_df, data.val_df, data.test_df
val_records, test_records = data.val_records, data.test_records
max_title_len = data.max_title_len


BT4222 Group 12: GATED Training & Comprehensive Evaluation (run tag: gated)
Config: {'tag': 'gated', 'eval_only': False, 'epochs': 3, 'batch_size': 128, 'lr': 0.0001, 'num_neg': 4, 'gamma': 2.0, 'pop_per_cat': 2, 'pop_extra': 10, 'pop_min_cat_clicks': 20, 'debug': False, 'model': 'gated', 'loss': 'ce', 'hist_dropout': 0.1, 'hist_trunc': 0.1, 'skip_seeds': False, 'skip_ablation': False}
Device: cuda (NVIDIA GeForce RTX 5060 Ti)


In [16]:
def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower()) if isinstance(text, str) else []


In [17]:
all_days, day2idx = data.all_days, data.day2idx
pop_lists = data.pop_lists
pop_sets = torch.as_tensor(data.pop_sets, dtype=torch.long, device=device)


In [18]:
def parse_history(raw):
    raw_h = raw.split() if isinstance(raw, str) else []
    hist = [news_id2idx[x] for x in raw_h if x in news_id2idx][-data.max_hist_len :]
    return [0] * (data.max_hist_len - len(hist)) + hist, len(raw_h)


In [19]:
def build_train_samples(df):
    # 1 positive + num_neg negatives sampled within each impression
    samples = []
    n_cold = 0
    for row in df.itertuples(index=False):
        hist, hist_len = parse_history(row.history)
        day = day2idx[row.time.normalize()]
        pos, neg = [], []
        for item in row.impressions.split():
            nid, lbl = item.split("-")
            if nid in news_id2idx:
                (pos if lbl == "1" else neg).append(news_id2idx[nid])
        if not pos or not neg:
            continue
        for p in pos:
            if len(neg) >= config.num_neg:
                sampled_neg = random.sample(neg, config.num_neg)
            else:
                sampled_neg = random.choices(neg, k=config.num_neg)
            samples.append((hist, [p] + sampled_neg, day))
            n_cold += hist_len == 0
    return samples, n_cold


In [20]:
model_cls = GatedNRMS if config.model == "gated" else NRMS


In [21]:
def make_model(use_self_attn=True):
    return model_cls(
        pretrained_embeddings,
        news_title_lookup,
        d_model=256,
        n_heads=16,
        query_dim=200,
        dropout=0.2,
        use_self_attn=use_self_attn,
        pop_sets=pop_sets,
    ).to(device)


In [22]:
def train_model(seed, ckpt_path, use_self_attn=True):
    """Trains for config.epochs epochs and keeps the checkpoint with the best validation NDCG@5."""
    set_seed(seed)
    samples, n_cold = build_train_samples(train_df)
    print(
        f"Constructed {len(samples)} training instances (1 pos + {config.num_neg} neg), "
        f"{n_cold} ({n_cold / max(len(samples), 1) * 100:.2f}%) with empty history."
    )
    loader = DataLoader(
        TrainDataset(samples, config.hist_dropout, config.hist_trunc),
        batch_size=config.batch_size,
        shuffle=True,
        drop_last=True,
    )
    model = make_model(use_self_attn)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    history = {
        "train_loss": [],
        "train_alpha": [],
        "val_auc": [],
        "val_mrr": [],
        "val_ndcg5": [],
        "val_ndcg10": [],
    }
    best_ndcg5 = -1.0
    for epoch in range(1, config.epochs + 1):
        model.train()
        total_loss, total_alpha = 0.0, 0.0
        t_epoch = time.time()
        for hist_b, cand_b, day_b in loader:
            hist_b, cand_b, day_b = (
                hist_b.to(device),
                cand_b.to(device),
                day_b.to(device),
            )
            optimizer.zero_grad()
            if config.model == "gated":
                scores, alpha = model(hist_b, cand_b, day_b, return_alpha=True)
                total_alpha += alpha.mean().item()
            else:
                scores = model(hist_b, cand_b, day_b)
            loss = ranking_loss(scores, config.loss, config.gamma)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(loader)
        avg_alpha = total_alpha / len(loader) if config.model == "gated" else 1.0
        epoch_time = time.time() - t_epoch
        t_val = time.time()
        val_metrics = evaluate_model(model, val_records, news_title_lookup, device)
        history["train_loss"].append(avg_loss)
        history["train_alpha"].append(avg_alpha)
        for k in ["auc", "mrr", "ndcg5", "ndcg10"]:
            history[f"val_{k}"].append(val_metrics[k])
        print(
            f"[seed {seed}] Epoch {epoch}/{config.epochs} | Train Loss: {avg_loss:.4f} | mean alpha: {avg_alpha:.3f} ({epoch_time:.1f}s) | "
            f"Val AUC: {val_metrics['auc']:.4f} | MRR: {val_metrics['mrr']:.4f} | "
            f"NDCG@5: {val_metrics['ndcg5']:.4f} | NDCG@10: {val_metrics['ndcg10']:.4f} ({time.time()-t_val:.1f}s)"
        )
        if val_metrics["ndcg5"] > best_ndcg5:
            best_ndcg5 = val_metrics["ndcg5"]
            torch.save(model.state_dict(), ckpt_path)
            print(f"  --> Saved new best checkpoint to {ckpt_path}")
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"Loaded best model from {ckpt_path} (Best Val NDCG@5: {best_ndcg5:.4f})")
    return model, history


In [23]:
prev = {}

if config.eval_only and os.path.exists(summary_path):
    with open(summary_path) as f:
        prev = json.load(f)

print(f"\n[4/7] Training {config.model.upper()} (Seed 42)...")
best_model_path = os.path.join(checkpoints_dir, f"{tag}_best_seed42.pt")

if config.eval_only and os.path.exists(best_model_path):
    print(f"--eval_only: loading checkpoint {best_model_path}")
    model = make_model()
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    history = prev.get("history", {})
else:
    model, history = train_model(42, best_model_path)

print(
    f"\n[5/7] Evaluating best model on Day 7 Test Partition ({len(test_records)} impressions)..."
)
t_test = time.time()
test_results = evaluate_model(model, test_records, news_title_lookup, device)
print(f"Test Benchmark Results (Seed 42):")
print(f"  • AUC:     {test_results['auc']:.4f}")
print(f"  • MRR:     {test_results['mrr']:.4f}")
print(f"  • NDCG@5:  {test_results['ndcg5']:.4f}")
print(f"  • NDCG@10: {test_results['ndcg10']:.4f}")
print(f"Evaluation completed in {time.time()-t_test:.2f}s.")
print("\n[6/7] Multi-Seed Robustness Evaluation (Seeds 42, 43, 44)...")
seed_results = {"seed42": metrics_of(test_results)}

if config.skip_seeds:
    print("Skipped (--skip_seeds).")
elif config.eval_only:
    seed_results = prev.get("multi_seed", {}).get("seeds", seed_results)
else:
    for extra_seed in [43, 44]:
        print(f"\n--- Running Seed {extra_seed} ---")
        m_seed, _ = train_model(
            extra_seed, os.path.join(checkpoints_dir, f"{tag}_best_seed{extra_seed}.pt")
        )
        seed_results[f"seed{extra_seed}"] = metrics_of(
            evaluate_model(m_seed, test_records, news_title_lookup, device)
        )
        r = seed_results[f"seed{extra_seed}"]
        print(
            f"Seed {extra_seed} Results: AUC={r['auc']:.4f}, MRR={r['mrr']:.4f}, NDCG@5={r['ndcg5']:.4f}, NDCG@10={r['ndcg10']:.4f}"
        )
        del m_seed
        torch.cuda.empty_cache()

aucs = [seed_results[s]["auc"] for s in seed_results]
mrrs = [seed_results[s]["mrr"] for s in seed_results]
n5s = [seed_results[s]["ndcg5"] for s in seed_results]
n10s = [seed_results[s]["ndcg10"] for s in seed_results]
print(f"\nMulti-Seed Summary (Mean ± Std over {len(seed_results)} seed(s)):")
print(f"  • AUC:     {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"  • MRR:     {np.mean(mrrs):.4f} ± {np.std(mrrs):.4f}")
print(f"  • NDCG@5:  {np.mean(n5s):.4f} ± {np.std(n5s):.4f}")
print(f"  • NDCG@10: {np.mean(n10s):.4f} ± {np.std(n10s):.4f}")
print("\n[7/7] Computing Cold-Start and Diurnal Temporal Stratifications...")
detailed = pd.DataFrame(test_results["detailed"])
cold_start = stratify_by_history(detailed)
print("\nUser Cold-Start Performance Breakdown:")



[4/7] Training GATED (Seed 42)...
Constructed 189519 training instances (1 pos + 4 neg), 3617 (1.91%) with empty history.
[seed 42] Epoch 1/3 | Train Loss: 1.4790 | mean alpha: 0.575 (86.0s) | Val AUC: 0.5993 | MRR: 0.2835 | NDCG@5: 0.2594 | NDCG@10: 0.3225 (3.4s)
  --> Saved new best checkpoint to /mnt/data/KHTN2023/Project/bt4222-group12/outputs/run_02/nrms/checkpoints/gated_best_seed42.pt
[seed 42] Epoch 2/3 | Train Loss: 1.3981 | mean alpha: 0.780 (76.7s) | Val AUC: 0.6304 | MRR: 0.3126 | NDCG@5: 0.2880 | NDCG@10: 0.3509 (3.4s)
  --> Saved new best checkpoint to /mnt/data/KHTN2023/Project/bt4222-group12/outputs/run_02/nrms/checkpoints/gated_best_seed42.pt
[seed 42] Epoch 3/3 | Train Loss: 1.3639 | mean alpha: 0.680 (76.8s) | Val AUC: 0.6403 | MRR: 0.3260 | NDCG@5: 0.3001 | NDCG@10: 0.3623 (3.4s)
  --> Saved new best checkpoint to /mnt/data/KHTN2023/Project/bt4222-group12/outputs/run_02/nrms/checkpoints/gated_best_seed42.pt
Loaded best model from /mnt/data/KHTN2023/Project/bt4222-g

In [24]:
for b in HIST_LABELS:
    s = cold_start[b]
    alpha_txt = (
        f", alpha={s['alpha_mean']:.3f}±{s['alpha_std']:.3f}"
        if config.model == "gated"
        else ""
    )
    print(
        f"  {b:<18} (N={s['count']:>5}): AUC={s['auc']:.4f}, MRR={s['mrr']:.4f}, NDCG@5={s['ndcg5']:.4f}, NDCG@10={s['ndcg10']:.4f}{alpha_txt}"
    )

hourly_summary = (
    detailed.groupby("hour")[["auc", "mrr", "ndcg5", "ndcg10"]].mean().reset_index()
)
print(
    f"Hourly NDCG@5 range: [{hourly_summary['ndcg5'].min():.4f}, {hourly_summary['ndcg5'].max():.4f}] (std {hourly_summary['ndcg5'].std():.4f}), "
    f"AUC range: [{hourly_summary['auc'].min():.4f}, {hourly_summary['auc'].max():.4f}] (std {hourly_summary['auc'].std():.4f})"
)
fallback = None

if config.model == "nrms":
    print(
        "\nEvaluating hard cold-start fallback (u = u_pop if |H| = 0, same popular sets as the gated model)..."
    )
    fb_results = evaluate_model(
        model, test_records, news_title_lookup, device, cold_fallback=True
    )
    fb_cold = stratify_by_history(pd.DataFrame(fb_results["detailed"]))
    fallback = {"overall": metrics_of(fb_results), "cold_start": fb_cold}
    cs_o, cs_f = cold_start[HIST_LABELS[0]], fb_cold[HIST_LABELS[0]]
    print(
        f"  Cold-start  NRMS:     AUC={cs_o['auc']:.4f}, MRR={cs_o['mrr']:.4f}, NDCG@5={cs_o['ndcg5']:.4f}, NDCG@10={cs_o['ndcg10']:.4f}"
    )
    print(
        f"  Cold-start  Fallback: AUC={cs_f['auc']:.4f}, MRR={cs_f['mrr']:.4f}, NDCG@5={cs_f['ndcg5']:.4f}, NDCG@10={cs_f['ndcg10']:.4f}"
    )
    print(
        f"  Overall     Fallback: AUC={fb_results['auc']:.4f}, MRR={fb_results['mrr']:.4f}, NDCG@5={fb_results['ndcg5']:.4f}, NDCG@10={fb_results['ndcg10']:.4f}"
    )

abl_results = None

if config.skip_ablation:
    print("\nAblation skipped (--skip_ablation).")
elif config.eval_only:
    abl_results = prev.get("ablation", {}).get("no_self_attn")
else:
    print("\nRunning Ablation: same model without Self-Attention...")
    ablation_model, _ = train_model(
        42,
        os.path.join(checkpoints_dir, f"{tag}_noattn_best_seed42.pt"),
        use_self_attn=False,
    )
    abl_results = metrics_of(
        evaluate_model(ablation_model, test_records, news_title_lookup, device)
    )
    print(
        f"Ablation (No Self-Attn) Results: AUC={abl_results['auc']:.4f}, MRR={abl_results['mrr']:.4f}, NDCG@5={abl_results['ndcg5']:.4f}, NDCG@10={abl_results['ndcg10']:.4f}"
    )
    del ablation_model
    torch.cuda.empty_cache()

print("\nGenerating publication figures...")
fig_path = lambda name: os.path.join(figures_dir, f"{file_prefix}{name}")
loss_name = "Focal Loss" if config.loss == "focal" else "Cross-Entropy Loss"


  Cold Start (0)     (N= 2214): AUC=0.5506, MRR=0.2867, NDCG@5=0.2732, NDCG@10=0.3279, alpha=0.387±0.000
  Sparse (1-4)       (N= 8092): AUC=0.5966, MRR=0.3380, NDCG@5=0.3390, NDCG@10=0.4000, alpha=0.506±0.133
  Warm (>=5)         (N=62846): AUC=0.6368, MRR=0.3361, NDCG@5=0.3176, NDCG@10=0.3816, alpha=0.679±0.101
Hourly NDCG@5 range: [0.2647, 0.3858] (std 0.0339), AUC range: [0.5962, 0.6711] (std 0.0252)

Running Ablation: same model without Self-Attention...


Constructed 189519 training instances (1 pos + 4 neg), 3617 (1.91%) with empty history.
[seed 42] Epoch 1/3 | Train Loss: 1.4464 | mean alpha: 0.819 (20.6s) | Val AUC: 0.6235 | MRR: 0.3126 | NDCG@5: 0.2895 | NDCG@10: 0.3499 (3.0s)
  --> Saved new best checkpoint to /mnt/data/KHTN2023/Project/bt4222-group12/outputs/run_02/nrms/checkpoints/gated_noattn_best_seed42.pt
[seed 42] Epoch 2/3 | Train Loss: 1.3780 | mean alpha: 0.819 (20.6s) | Val AUC: 0.6301 | MRR: 0.3226 | NDCG@5: 0.2965 | NDCG@10: 0.3572 (3.0s)
  --> Saved new best checkpoint to /mnt/data/KHTN2023/Project/bt4222-group12/outputs/run_02/nrms/checkpoints/gated_noattn_best_seed42.pt
[seed 42] Epoch 3/3 | Train Loss: 1.3518 | mean alpha: 0.723 (20.5s) | Val AUC: 0.6395 | MRR: 0.3272 | NDCG@5: 0.3030 | NDCG@10: 0.3635 (3.0s)
  --> Saved new best checkpoint to /mnt/data/KHTN2023/Project/bt4222-group12/outputs/run_02/nrms/checkpoints/gated_noattn_best_seed42.pt
Loaded best model from /mnt/data/KHTN2023/Project/bt4222-group12/outputs

## Validation by history group

Score the seed-42 checkpoint on the validation day and split it by history length, so validation and test can be compared per user group.


In [25]:
val_results = evaluate_model(model, val_records, news_title_lookup, device)
val_by_group = stratify_by_history(pd.DataFrame(val_results["detailed"]))
print(f"Validation (seed 42): AUC={val_results['auc']:.4f}, MRR={val_results['mrr']:.4f}, NDCG@5={val_results['ndcg5']:.4f}, NDCG@10={val_results['ndcg10']:.4f}")
for b in HIST_LABELS:
    s = val_by_group[b]
    print(
        f"  {b:<18} (N={s['count']:>5}): AUC={s['auc']:.4f}, MRR={s['mrr']:.4f}, NDCG@5={s['ndcg5']:.4f}, NDCG@10={s['ndcg10']:.4f}"
    )


Validation (seed 42): AUC=0.6403, MRR=0.3260, NDCG@5=0.3001, NDCG@10=0.3624
  Cold Start (0)     (N=  772): AUC=0.5818, MRR=0.2728, NDCG@5=0.2498, NDCG@10=0.3104
  Sparse (1-4)       (N= 3247): AUC=0.6096, MRR=0.3158, NDCG@5=0.3048, NDCG@10=0.3655
  Warm (>=5)         (N=26251): AUC=0.6458, MRR=0.3288, NDCG@5=0.3010, NDCG@10=0.3635


In [26]:
if history.get("train_loss"):
    ep_range = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(
        ep_range,
        history["train_loss"],
        marker="o",
        color="#2b5c8f",
        lw=2.5,
        label="Train Loss",
    )
    axes[0].set_title(f"Training {loss_name} vs. Epoch", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Epoch", fontsize=12)
    axes[0].set_ylabel(loss_name, fontsize=12)
    axes[0].set_xticks(list(ep_range))
    axes[0].legend(frameon=True)
    axes[0].grid(True, linestyle="--", alpha=0.6)
    axes[1].plot(
        ep_range,
        history["val_auc"],
        marker="s",
        color="#2ca02c",
        lw=2,
        label="Validation AUC",
    )
    axes[1].plot(
        ep_range,
        history["val_ndcg5"],
        marker="^",
        color="#d62728",
        lw=2,
        label="Validation NDCG@5",
    )
    axes[1].plot(
        ep_range,
        history["val_mrr"],
        marker="d",
        color="#9467bd",
        lw=2,
        label="Validation MRR",
    )
    axes[1].plot(
        ep_range,
        history["val_ndcg10"],
        marker="v",
        color="#ff7f0e",
        lw=2,
        label="Validation NDCG@10",
    )
    axes[1].set_title(
        "Validation Ranking Performance on Day 6", fontsize=13, fontweight="bold"
    )
    axes[1].set_xlabel("Epoch", fontsize=12)
    axes[1].set_ylabel("Metric Score", fontsize=12)
    axes[1].set_xticks(list(ep_range))
    axes[1].legend(frameon=True)
    axes[1].grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(fig_path("training_curves.png"), dpi=300)
    plt.close()

fig, ax = plt.subplots(figsize=(10, 5.5))
x = np.arange(len(HIST_LABELS))
width = 0.2

for off, key, name, color in [
    (-1.5, "auc", "AUC", "#1f77b4"),
    (-0.5, "mrr", "MRR", "#ff7f0e"),
    (0.5, "ndcg5", "NDCG@5", "#2ca02c"),
    (1.5, "ndcg10", "NDCG@10", "#d62728"),
]:
    ax.bar(
        x + off * width,
        [cold_start[b][key] for b in HIST_LABELS],
        width,
        label=name,
        color=color,
    )

ax.set_title(
    f"{config.model.upper()} Ranking Performance across User History Lengths",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlabel("User Interaction History Length", fontsize=12)
ax.set_ylabel("Metric Score", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(HIST_LABELS, fontsize=11)
ax.set_ylim(0.15, 0.75)
ax.legend(frameon=True, loc="upper left")
ax.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig(fig_path("cold_start_analysis.png"), dpi=300)
plt.close()
fig, ax1 = plt.subplots(figsize=(12, 5.5))
color1 = "#1f77b4"
ax1.set_xlabel("Hour of Day (UTC)", fontsize=12)
ax1.set_ylabel("NDCG@5", color=color1, fontsize=12)
line1 = ax1.plot(
    hourly_summary["hour"],
    hourly_summary["ndcg5"],
    color=color1,
    marker="o",
    lw=2.5,
    label="NDCG@5",
)
ax1.tick_params(axis="y", labelcolor=color1)
ax1.set_xticks(range(0, 24))
ax1.grid(True, linestyle="--", alpha=0.6)
ax2 = ax1.twinx()


In [27]:
color2 = "#d62728"
ax2.set_ylabel("AUC", color=color2, fontsize=12)
line2 = ax2.plot(
    hourly_summary["hour"],
    hourly_summary["auc"],
    color=color2,
    marker="s",
    lw=2.5,
    linestyle="--",
    label="AUC",
)
ax2.tick_params(axis="y", labelcolor=color2)
lines = line1 + line2
ax1.legend(lines, [l.get_label() for l in lines], loc="lower right", frameon=True)
plt.title(
    "Diurnal Ranking Performance Stability across Day 7 Test Hours",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(fig_path("temporal_hourly_analysis.png"), dpi=300)
plt.close()

if abl_results is not None:
    fig, ax = plt.subplots(figsize=(8, 5))
    metrics_names = ["AUC", "MRR", "NDCG@5", "NDCG@10"]
    full_vals = [
        test_results["auc"],
        test_results["mrr"],
        test_results["ndcg5"],
        test_results["ndcg10"],
    ]
    abl_vals = [
        abl_results["auc"],
        abl_results["mrr"],
        abl_results["ndcg5"],
        abl_results["ndcg10"],
    ]
    x = np.arange(len(metrics_names))
    width = 0.35
    ax.bar(
        x - width / 2,
        full_vals,
        width,
        label="Full (Multi-Head Self-Attn)",
        color="#2b5c8f",
    )
    ax.bar(
        x + width / 2,
        abl_vals,
        width,
        label="Ablation (Without Self-Attn)",
        color="#d95f02",
    )
    ax.set_title(
        "Ablation Study: Contribution of Multi-Head Self-Attention",
        fontsize=13,
        fontweight="bold",
    )
    ax.set_ylabel("Score", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_names, fontsize=11)
    ax.set_ylim(0.2, 0.75)
    for i in range(len(metrics_names)):
        ax.text(
            x[i] - width / 2,
            full_vals[i] + 0.01,
            f"{full_vals[i]:.4f}",
            ha="center",
            fontsize=9,
            fontweight="bold",
        )
        ax.text(
            x[i] + width / 2,
            abl_vals[i] + 0.01,
            f"{abl_vals[i]:.4f}",
            ha="center",
            fontsize=9,
            fontweight="bold",
        )
    ax.legend(frameon=True)
    ax.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(fig_path("ablation_study.png"), dpi=300)
    plt.close()

sample_news_ids = list(all_news["news_id"].iloc[10:14])
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
model.eval()


GatedNRMS(
  (news_encoder): NewsEncoder(
    (embedding): Embedding(21351, 300, padding_idx=0)
    (dropout): Dropout(p=0.2, inplace=False)
    (proj): Linear(in_features=300, out_features=256, bias=True)
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (pooling): AttentionPooling(
      (proj): Linear(in_features=256, out_features=200, bias=True)
    )
  )
  (user_encoder): UserEncoder(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
    )
    (dropout): Dropout(p=0.2, inplace=False)
    (pooling): AttentionPooling(
      (proj): Linear(in_features=256, out_features=200, bias=True)
    )
  )
  (gate): Sequential(
    (0): Linear(in_features=513, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [28]:
for idx, snid in enumerate(sample_news_ids):
    n_idx = news_id2idx[snid]
    toks = news_title_lookup[n_idx : n_idx + 1]
    with torch.no_grad():
        _, alpha = model.news_encoder(toks, return_weights=True)
    alpha_np = alpha[0].detach().cpu().numpy()
    title = all_news[all_news["news_id"] == snid]["title"].values[0]
    title_words = tokenize(title)[:max_title_len]
    alpha_words = alpha_np[: len(title_words)]
    if alpha_words.sum() > 0:
        alpha_words = alpha_words / alpha_words.sum()
    y_pos = np.arange(len(title_words))
    axes[idx].barh(y_pos, alpha_words, color="#2b5c8f", alpha=0.85)
    axes[idx].set_yticks(y_pos)
    axes[idx].set_yticklabels(title_words, fontsize=10)
    axes[idx].invert_yaxis()
    axes[idx].set_title(f'"{title[:40]}..."', fontsize=11, fontweight="bold")
    axes[idx].set_xlabel("Additive Attention Weight", fontsize=10)
    axes[idx].grid(True, linestyle="--", alpha=0.5)

plt.suptitle(
    "Word-Level Additive Attention Weights on Candidate News Titles",
    fontsize=14,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig(fig_path("attention_visualization.png"), dpi=300)
plt.close()

if fallback is not None:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    metrics_cs = ["AUC", "MRR", "NDCG@5", "NDCG@10"]
    keys = ["auc", "mrr", "ndcg5", "ndcg10"]
    orig_vals = [cold_start[HIST_LABELS[0]][k] for k in keys]
    fb_vals = [fallback["cold_start"][HIST_LABELS[0]][k] for k in keys]
    x = np.arange(len(metrics_cs))
    width = 0.35
    ax.bar(
        x - width / 2,
        orig_vals,
        width,
        label="Original NRMS",
        color="#e41a1c",
        alpha=0.85,
    )
    ax.bar(
        x + width / 2,
        fb_vals,
        width,
        label="NRMS + Hard Popularity Fallback",
        color="#2ca02c",
        alpha=0.85,
    )
    ax.set_title(
        "Cold-Start Users: Original NRMS vs. Hard Popularity Fallback",
        fontsize=13,
        fontweight="bold",
    )
    ax.set_ylabel("Score", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_cs, fontsize=11)
    ax.set_ylim(0.15, 0.7)
    for i in range(len(metrics_cs)):
        ax.text(
            x[i] - width / 2,
            orig_vals[i] + 0.01,
            f"{orig_vals[i]:.4f}",
            ha="center",
            fontsize=9,
            fontweight="bold",
        )
        ax.text(
            x[i] + width / 2,
            fb_vals[i] + 0.01,
            f"{fb_vals[i]:.4f}",
            ha="center",
            fontsize=9,
            fontweight="bold",
        )
    ax.legend(frameon=True, loc="upper left")
    ax.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(fig_path("cold_start_mitigation.png"), dpi=300)
    plt.close()


In [29]:
if config.model == "gated":
    fig, ax = plt.subplots(figsize=(9, 5))
    means = [cold_start[b]["alpha_mean"] for b in HIST_LABELS]
    stds = [cold_start[b]["alpha_std"] for b in HIST_LABELS]
    x = np.arange(len(HIST_LABELS))
    ax.bar(
        x,
        means,
        yerr=stds,
        capsize=5,
        color="#2b5c8f",
        alpha=0.85,
        label="History-branch weight α",
    )
    ax.bar(
        x,
        [1 - m for m in means],
        bottom=means,
        color="#ff7f0e",
        alpha=0.6,
        label="Popularity-branch weight 1 âˆ’ α",
    )
    for i, m in enumerate(means):
        ax.text(
            x[i],
            m / 2,
            f"{m:.2f}",
            ha="center",
            va="center",
            color="white",
            fontsize=10,
            fontweight="bold",
        )
    ax.set_xticks(x)
    ax.set_xticklabels(HIST_LABELS, fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Gate weight", fontsize=12)
    ax.set_title(
        "Learned Gate: Branch Contribution by User History Length (Day 7)",
        fontsize=13,
        fontweight="bold",
    )
    ax.legend(frameon=True, loc="lower right")
    ax.grid(True, linestyle="--", alpha=0.6, axis="y")
    plt.tight_layout()
    plt.savefig(fig_path("gate_alpha_by_history.png"), dpi=300)
    plt.close()

results_summary = {
    "config": {**vars(config), "tag": tag},
    "popular_sets": {str(d.date()): len(p) for d, p in zip(all_days, pop_lists)},
    "single_seed": metrics_of(test_results),
    "multi_seed": {
        "seeds": seed_results,
        "mean": {
            "auc": float(np.mean(aucs)),
            "mrr": float(np.mean(mrrs)),
            "ndcg5": float(np.mean(n5s)),
            "ndcg10": float(np.mean(n10s)),
        },
        "std": {
            "auc": float(np.std(aucs)),
            "mrr": float(np.std(mrrs)),
            "ndcg5": float(np.std(n5s)),
            "ndcg10": float(np.std(n10s)),
        },
    },
    "cold_start": cold_start,
    "hourly": hourly_summary.to_dict(orient="list"),
    "history": history,
}

if fallback is not None:
    results_summary["hard_fallback"] = fallback

if abl_results is not None:
    results_summary["ablation"] = {
        "full": metrics_of(test_results),
        "no_self_attn": abl_results,
    }

with open(summary_path, "w") as f:
    json.dump(results_summary, f, indent=2)

print(f"Saved summary to {summary_path}")
print("\nAll experiments, evaluations, and figures successfully finished and saved!")
print("=" * 70)


Saved summary to /mnt/data/KHTN2023/Project/bt4222-group12/outputs/run_02/nrms/figures/results_summary_gated.json

All experiments, evaluations, and figures successfully finished and saved!


## 14. Paired multi-seed comparison

Compare measured NRMS, hard-fallback and gated checkpoints from seeds 42, 43 and 44. Run the reference and gated configurations with the same prepared inputs before enabling this section.


In [30]:
from scipy.stats import wilcoxon

METRICS = ["auc", "mrr", "ndcg5", "ndcg10"]
METRIC_NAMES = {"auc": "AUC", "mrr": "MRR", "ndcg5": "NDCG@5", "ndcg10": "NDCG@10"}
SEEDS = [42, 43, 44]
VARIANTS = ["nrms", "fallback", "gated"]
VARIANT_NAMES = {
    "nrms": "Original NRMS",
    "fallback": "NRMS + hard fallback",
    "gated": "Gated NRMS",
}
# Reference categorical palette, slots 1-3 (validated all-pairs for CVD); text stays in neutral ink.
COLORS = {"nrms": "#2a78d6", "fallback": "#eb6834", "gated": "#1baf7a"}
INK, INK_2, GRID = "#0b0b0b", "#52514e", "#e4e3df"


In [31]:
def bootstrap_ci(diff, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(diff), size=(n_boot, len(diff)))
    means = diff[idx].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


In [32]:
def paired_test(a, b, per_seed_a, per_seed_b, mask):
    """a, b: seed-averaged per-impression scores; per_seed_*: [n_seeds, N]. Tests mean(b - a) on mask."""
    diff = b[mask] - a[mask]
    nz = diff[diff != 0]
    p = float(wilcoxon(nz).pvalue) if len(nz) > 0 else 1.0
    lo, hi = bootstrap_ci(diff)
    seed_diffs = [
        float((sb[mask] - sa[mask]).mean()) for sa, sb in zip(per_seed_a, per_seed_b)
    ]
    return {
        "n": int(mask.sum()),
        "mean_diff": float(diff.mean()),
        "ci95": [lo, hi],
        "wilcoxon_p": p,
        "per_seed_diff": seed_diffs,
    }


In [33]:
def plot_summary(summary, figures_dir):
    x = np.arange(len(HIST_LABELS))
    # 3. Figure: NDCG@5 by history bin (3 variants, mean ± std over seeds) + gate weight alpha by bin
    fig, axes = plt.subplots(
        1, 2, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1.6, 1]}
    )
    ax = axes[0]
    offset = 0.22
    for k, v in enumerate(VARIANTS):
        means = [summary["groups"][b][v]["ndcg5"]["mean"] for b in HIST_LABELS]
        stds = [summary["groups"][b][v]["ndcg5"]["std"] for b in HIST_LABELS]
        ax.errorbar(
            x + (k - 1) * offset,
            means,
            yerr=stds,
            fmt="o",
            markersize=8,
            capsize=3,
            color=COLORS[v],
            ecolor=COLORS[v],
            elinewidth=2,
            markeredgecolor="#fcfcfb",
            markeredgewidth=1.5,
            label=VARIANT_NAMES[v],
        )
    for k, v in [(0, "nrms"), (2, "gated")]:
        m = summary["groups"][HIST_LABELS[0]][v]["ndcg5"]["mean"]
        ax.text(
            x[0] + (k - 1) * offset + 0.08,
            m,
            f"{m:.3f}",
            ha="left",
            va="center",
            fontsize=9,
            color=INK,
        )
    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{b}\nN={summary['groups'][b]['count']:,}" for b in HIST_LABELS],
        fontsize=9.5,
        color=INK_2,
    )
    ax.set_ylim(0.21, 0.37)
    ax.set_xlim(-0.5, len(HIST_LABELS) - 0.5)
    ax.set_ylabel("NDCG@5 (mean ± std, 3 seeds)", color=INK_2)
    ax.set_title(
        "NDCG@5 by user history length (Day 7 test)",
        fontsize=12.5,
        fontweight="bold",
        color=INK,
        loc="left",
    )
    ax.legend(frameon=False, loc="lower right", fontsize=9.5)
    ax = axes[1]
    am = [summary["groups"][b]["gated"]["alpha"]["mean"] for b in HIST_LABELS]
    asd = [summary["groups"][b]["gated"]["alpha"]["std"] for b in HIST_LABELS]
    ax.bar(
        x,
        am,
        0.6,
        yerr=asd,
        capsize=3,
        color=COLORS["gated"],
        error_kw={"ecolor": INK_2, "elinewidth": 1},
    )
    for i, m in enumerate(am):
        ax.text(
            x[i], m + asd[i] + 0.02, f"{m:.2f}", ha="center", fontsize=9.5, color=INK
        )
    ax.set_xticks(x)
    ax.set_xticklabels(
        [b.replace(" (", "\n(") for b in HIST_LABELS], fontsize=9.5, color=INK_2
    )
    ax.set_ylim(0, 1)
    ax.set_ylabel("History-branch weight α", color=INK_2)
    ax.set_title(
        "Gated NRMS: learned weight α (1 âˆ’ α â†’ popularity)",
        fontsize=12.5,
        fontweight="bold",
        color=INK,
        loc="left",
    )
    for ax in axes:
        ax.grid(True, axis="y", color=GRID, linewidth=0.8)
        ax.grid(False, axis="x")
        ax.set_axisbelow(True)
        for side in ["top", "right"]:
            ax.spines[side].set_visible(False)
        for side in ["left", "bottom"]:
            ax.spines[side].set_color(GRID)
        ax.tick_params(colors=INK_2)
    plt.tight_layout()
    out_fig = os.path.join(figures_dir, "seed_comparison_by_history.png")
    plt.savefig(out_fig, dpi=300, facecolor="#fcfcfb")
    plt.close()
    print(f"Saved {out_fig}")


In [34]:
def compare_seed_runs(nrms_tag="nrms", gated_tag="gated"):
    test_records = data.test_records
    hist_len = np.array([r["history_len"] for r in test_records])
    bins = pd.cut(hist_len, bins=HIST_BINS, labels=HIST_LABELS)
    cold = hist_len == 0

    def make(cls):
        return cls(
            pretrained_embeddings,
            news_title_lookup,
            d_model=256,
            n_heads=16,
            query_dim=200,
            dropout=0.2,
            pop_sets=pop_sets,
        ).to(device)

    # scores[variant][metric] -> [n_seeds, N]; alpha[variant] -> [n_seeds, N]
    scores = {v: {m: [] for m in METRICS} for v in VARIANTS}
    alpha = {v: [] for v in VARIANTS}
    for seed in SEEDS:
        for cls, tag, runs in [
            (NRMS, nrms_tag, [("nrms", False), ("fallback", True)]),
            (GatedNRMS, gated_tag, [("gated", False)]),
        ]:
            ckpt = os.path.join(checkpoints_dir, f"{tag}_best_seed{seed}.pt")
            model = make(cls)
            model.load_state_dict(torch.load(ckpt, map_location=device))
            for variant, fb in runs:
                res = evaluate_model(
                    model, test_records, news_title_lookup, device, cold_fallback=fb
                )
                det = pd.DataFrame(res["detailed"])
                for m in METRICS:
                    scores[variant][m].append(det[m].to_numpy())
                alpha[variant].append(det["alpha"].to_numpy())
                print(
                    f"seed {seed} | {VARIANT_NAMES[variant]:<22} | "
                    + " | ".join(f"{METRIC_NAMES[m]} {res[m]:.4f}" for m in METRICS)
                )
            del model
            torch.cuda.empty_cache()
    scores = {v: {m: np.stack(scores[v][m]) for m in METRICS} for v in VARIANTS}
    alpha = {v: np.stack(alpha[v]) for v in VARIANTS}

    # 1. Mean ± std over seeds, overall and per history bin
    def seed_stats(arr, mask):
        per_seed = arr[:, mask].mean(axis=1)
        return {
            "mean": float(per_seed.mean()),
            "std": float(per_seed.std()),
            "per_seed": per_seed.tolist(),
        }

    groups = {
        "Overall": np.ones(len(hist_len), dtype=bool),
        "Users with history (>=1)": ~cold,
    }
    groups.update({b: (bins == b) for b in HIST_LABELS})
    summary = {"seeds": SEEDS, "groups": {}}
    for g, mask in groups.items():
        summary["groups"][g] = {"count": int(mask.sum())}
        for v in VARIANTS:
            entry = {m: seed_stats(scores[v][m], mask) for m in METRICS}
            if v == "gated":
                entry["alpha"] = seed_stats(alpha[v], mask)
            summary["groups"][g][v] = entry
    print("\n" + "=" * 100)
    print("MEAN ± STD OVER SEEDS 42/43/44 (Day 7 test)")
    print("=" * 100)
    for g in groups:
        s = summary["groups"][g]
        print(f"\n{g} (N={s['count']})")
        for v in VARIANTS:
            row = " | ".join(
                f"{METRIC_NAMES[m]} {s[v][m]['mean']:.4f}±{s[v][m]['std']:.4f}"
                for m in METRICS
            )
            extra = (
                f" | alpha {s[v]['alpha']['mean']:.3f}±{s[v]['alpha']['std']:.3f}"
                if v == "gated"
                else ""
            )
            print(f"  {VARIANT_NAMES[v]:<22} {row}{extra}")
    # 2. Paired significance tests on seed-averaged per-impression scores
    comparisons = [("nrms", "gated"), ("fallback", "gated"), ("nrms", "fallback")]
    test_groups = {
        "Overall": groups["Overall"],
        "Cold Start (0)": cold,
        "Users with history (>=1)": ~cold,
    }
    summary["paired_tests"] = {}
    print("\n" + "=" * 100)
    print(
        "PAIRED TESTS: mean(B - A) per impression, seed-averaged | 95% bootstrap CI | Wilcoxon p | per-seed mean diff"
    )
    print("=" * 100)
    for a, b in comparisons:
        key = f"{b}_vs_{a}"
        summary["paired_tests"][key] = {}
        for g, mask in test_groups.items():
            summary["paired_tests"][key][g] = {}
            for m in METRICS:
                r = paired_test(
                    scores[a][m].mean(axis=0),
                    scores[b][m].mean(axis=0),
                    scores[a][m],
                    scores[b][m],
                    mask,
                )
                summary["paired_tests"][key][g][m] = r
                print(
                    f"{VARIANT_NAMES[b]:>20} vs {VARIANT_NAMES[a]:<20} | {g:<15} (N={r['n']:>5}) | {METRIC_NAMES[m]:<7} "
                    f"Δ={r['mean_diff']:+.4f} CI[{r['ci95'][0]:+.4f}, {r['ci95'][1]:+.4f}] p={r['wilcoxon_p']:.2e} "
                    f"seeds=[{', '.join(f'{d:+.4f}' for d in r['per_seed_diff'])}]"
                )
    out_json = os.path.join(figures_dir, "seed_comparison_summary.json")
    with open(out_json, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nSaved {out_json}")
    plot_summary(summary, figures_dir)


In [35]:
def _have_checkpoints(tag, seeds=(42, 43, 44)):
    return all(
        os.path.exists(os.path.join(checkpoints_dir, f"{tag}_best_seed{s}.pt"))
        for s in seeds
    )


# Runs automatically once both the "nrms" and "gated" runs have finished.
RUN_PAIRED_COMPARISON = _have_checkpoints("nrms") and _have_checkpoints("gated")
print("Paired comparison:", "running" if RUN_PAIRED_COMPARISON
      else "skipped (run this notebook with NRMS_VARIANT='nrms' and 'gated' first)")

if RUN_PAIRED_COMPARISON:
    compare_seed_runs()


Paired comparison: skipped (run this notebook with NRMS_VARIANT='nrms' and 'gated' first)


## Global top-20 popularity fallback

Evaluate the original frozen-training top-20 popularity prior for users with empty history. This post-hoc reference differs from the daily category-stratified learned gate. A matching plain-NRMS checkpoint is required.


In [36]:
@torch.no_grad()
def evaluate_global_popularity_fallback(reference_model):
    reference_model.eval()
    news_vecs = torch.zeros(len(news_title_lookup), 256, device=device)
    for start in range(0, len(news_title_lookup), 2048):
        news_vecs[start : start + 2048] = reference_model.news_encoder(
            news_title_lookup[start : start + 2048]
        )
    news_vecs[0] = 0
    top_pop_tensor = torch.as_tensor(
        data.top_popular_ids, dtype=torch.long, device=device
    )
    u_pop = reference_model.user_encoder(
        news_vecs[top_pop_tensor].unsqueeze(0)
    ).squeeze(0)
    histories = torch.as_tensor(
        [record["history"] for record in test_records], dtype=torch.long, device=device
    )
    user_vecs = torch.zeros(len(test_records), 256, device=device)
    for start in range(0, len(test_records), 1024):
        batch = histories[start : start + 1024]
        user_vecs[start : start + 1024] = reference_model.user_encoder(
            news_vecs[batch], mask=batch > 0
        )
    rows = []
    for index, record in enumerate(test_records):
        candidates = news_vecs[record["candidates"]]
        user = u_pop if record["history_len"] == 0 else user_vecs[index]
        original = eval_impression(
            record["labels"], torch.mv(candidates, user_vecs[index]).cpu().numpy()
        )
        fallback = eval_impression(
            record["labels"], torch.mv(candidates, user).cpu().numpy()
        )
        for variant, values in [
            ("nrms", original),
            ("global_top20_fallback", fallback),
        ]:
            rows.append(
                {
                    "impression_id": record["impression_id"],
                    "history_len": record["history_len"],
                    "variant": variant,
                    **dict(zip(["auc", "mrr", "ndcg5", "ndcg10"], values)),
                }
            )
    return pd.DataFrame(rows)


In [37]:
# Needs the plain NRMS seed-42 checkpoint (NRMS_VARIANT = "nrms").
RUN_GLOBAL_POPULARITY_FALLBACK = _have_checkpoints("nrms", seeds=(42,))

if RUN_GLOBAL_POPULARITY_FALLBACK:
    reference_model = NRMS(
        pretrained_embeddings,
        news_title_lookup,
        d_model=256,
        n_heads=16,
        query_dim=200,
        dropout=0.2,
    ).to(device)
    reference_model.load_state_dict(
        torch.load(
            os.path.join(checkpoints_dir, "nrms_best_seed42.pt"), map_location=device
        )
    )
    global_fallback = evaluate_global_popularity_fallback(reference_model)
    global_fallback.to_csv(
        os.path.join(figures_dir, "global_top20_fallback.csv"), index=False
    )
    display(
        global_fallback.groupby("variant")[["auc", "mrr", "ndcg5", "ndcg10"]].mean()
    )
